# Mercury â€” Fine-Tuning Notebook (all 6 catalog models)

Covers every model in `Backend/tiers.py`'s catalog, not just Parakeet/CosyVoice2:

**ASR**: Distil-Whisper-Large-v3 (Free tier) Â· Whisper-Large-v3-Turbo (Pro) Â· Parakeet-TDT-0.6B-v2 (Max/Enterprise)
**TTS**: Kokoro-82M (Free) Â· Bark (Pro) Â· CosyVoice2-0.5B (Max/Enterprise)

Fine-tuning is intentionally deferred ("very later" per project direction) â€” this notebook is
scaffolded-but-inert: installs, dataset loading, and LoRA config are wired up per model, but
training cells are not run until real fine-tuning data is ready. Recommended method for all
6 models: **LoRA/PEFT, not full fine-tune** â€” see `MEMORY.md`'s fine-tuning-method section for why
(Kaggle's single T4, 16GB VRAM, 30hr/week cap makes full fine-tune risky/slow; LoRA fits comfortably).

Datasets: point `DATASET_DIR` below at your prepared data. Audio is stored as MP3, not WAV, to
save space â€” if your source data is WAV, convert first (see the conversion cell below).

## 0. Setup â€” install deps for all 6 models

In [ ]:
!pip install -q nemo_toolkit[asr]==2.0.0 transformers==4.47.1 torch==2.5.1 \
    peft accelerate kokoro==0.7.16 soundfile pydub jiwer huggingface_hub

# CosyVoice2 isn't a pip package â€” clone its repo for the fine-tune recipe it ships.
!git clone -q https://github.com/FunAudioLLM/CosyVoice.git /kaggle/working/CosyVoice || true


## 1. Convert any WAV source audio to MP3

Run once against your raw dataset directory before building manifests â€” keeps Kaggle's disk
quota and any exported archives small. Skip if your data is already MP3.

In [ ]:
import os
from pydub import AudioSegment

DATASET_DIR = "/kaggle/input/your-dataset"  # <-- point at your real dataset
OUT_DIR = "/kaggle/working/audio_mp3"
os.makedirs(OUT_DIR, exist_ok=True)

converted = 0
for root, _, files in os.walk(DATASET_DIR):
    for f in files:
        if f.lower().endswith(".wav"):
            src = os.path.join(root, f)
            dst = os.path.join(OUT_DIR, os.path.splitext(f)[0] + ".mp3")
            AudioSegment.from_wav(src).export(dst, format="mp3", bitrate="128k")
            converted += 1
print(f"Converted {converted} wav files to mp3 in {OUT_DIR}")


## 2. Build train/val manifests

NeMo-format manifests (`{"audio_filepath": ..., "text": ..., "duration": ...}` per line) work for
the NeMo model directly; the HF-transformers models (Whisper variants) instead want a
`datasets.Dataset` â€” both are built from the same manifest here.

In [ ]:
import json
import soundfile as sf

MANIFEST_TRAIN = "/kaggle/working/train_manifest.json"
MANIFEST_VAL = "/kaggle/working/val_manifest.json"

def build_manifest(audio_text_pairs, out_path):
    with open(out_path, "w") as f:
        for audio_path, text in audio_text_pairs:
            info = sf.info(audio_path)
            f.write(json.dumps({
                "audio_filepath": audio_path,
                "text": text,
                "duration": info.duration,
            }) + "\n")

# TODO: populate from your real (audio_path, transcript) pairs once data is ready.
train_pairs = []
val_pairs = []
build_manifest(train_pairs, MANIFEST_TRAIN)
build_manifest(val_pairs, MANIFEST_VAL)
print(f"train={len(train_pairs)} val={len(val_pairs)} examples (0 until real data is wired in)")


---
## 3. ASR â€” Distil-Whisper-Large-v3 (Free tier)

Smallest/fastest ASR model â€” good first target to validate the fine-tuning pipeline works
end-to-end before spending Kaggle hours on the bigger models below.

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import LoraConfig, get_peft_model

MODEL_ID = "distil-whisper/distil-large-v3"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training loop intentionally not run yet â€” wire up a Seq2SeqTrainer against the
# manifests above once train_pairs/val_pairs are populated with real data.


## 4. ASR â€” Whisper-Large-v3-Turbo (Pro tier)

In [ ]:
MODEL_ID = "openai/whisper-large-v3-turbo"
processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 5. ASR â€” Parakeet-TDT-0.6B-v2 (Max/Enterprise tier)

Uses NeMo's native PEFT/adapter support rather than a hand-rolled LoRA wrapper â€” avoids the
exact plumbing bugs v2's custom `lora_trainer.py` had (missing `total_step`, undefined vars,
etc. â€” see `MEMORY.md`'s BUG-001/BUG-002).

In [ ]:
import nemo.collections.asr as nemo_asr

asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v2")

# NeMo's adapter/PEFT config â€” attach once manifests are real; see
# nemo.collections.common.parts.adapter_modules for the adapter types available.
# asr_model.add_adapter(name="lora_medical", cfg=<AdapterConfig>)

print(asr_model.cfg.train_ds if hasattr(asr_model.cfg, "train_ds") else "no train_ds in cfg yet")


---
## 6. TTS â€” Kokoro-82M (Free tier)

Kokoro is small enough that LoRA may be unnecessary â€” a light full fine-tune on a small
medical-vocabulary + accent dataset is plausible within Kaggle's budget. Evaluate both.

In [ ]:
from kokoro import KPipeline

pipeline = KPipeline(lang_code="a")
# Kokoro doesn't ship a first-party fine-tune recipe as of this writing â€” community
# recipes exist (search "kokoro-82m fine-tune lora" on HF/GitHub) but aren't vetted here yet.


## 7. TTS â€” Bark (Pro tier)

In [ ]:
from transformers import AutoProcessor, BarkModel
from peft import LoraConfig, get_peft_model

MODEL_ID = "suno/bark"
bark_processor = AutoProcessor.from_pretrained(MODEL_ID)
bark_model = BarkModel.from_pretrained(MODEL_ID)

# Bark's semantic/coarse/fine sub-models each need their own LoRA target modules â€”
# start with the semantic model only, since it's the one that carries linguistic content.
lora_config = LoraConfig(r=16, lora_alpha=32, target_modules=["c_attn"], lora_dropout=0.05, bias="none")
bark_model.semantic = get_peft_model(bark_model.semantic, lora_config)
bark_model.semantic.print_trainable_parameters()


## 8. TTS â€” CosyVoice2-0.5B (Max/Enterprise tier)

Official repo ships a fine-tune recipe (cloned in step 0) â€” check whether it exposes a
LoRA/adapter mode; if only full-param fine-tuning is offered upstream, freeze most layers
manually and train only the last few blocks as a manual low-rank-equivalent.

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/CosyVoice")
sys.path.insert(0, "/kaggle/working/CosyVoice/third_party/Matcha-TTS")

from cosyvoice.cli.cosyvoice import CosyVoice2

cosyvoice = CosyVoice2("FunAudioLLM/CosyVoice2-0.5B")
# See CosyVoice/examples/ for the official fine-tune recipe once real data is ready.


---
## 9. Evaluation harness (run after any fine-tune, before promoting a model)

WER/CER on a held-out set, compared against the current production checkpoint â€” don't promote
a fine-tuned model unless it beats the baseline on this set.

In [ ]:
from jiwer import wer, cer

def evaluate(model_transcribe_fn, val_pairs):
    refs, hyps = [], []
    for audio_path, ref_text in val_pairs:
        hyps.append(model_transcribe_fn(audio_path))
        refs.append(ref_text)
    if not refs:
        return {"wer": None, "cer": None, "n": 0}
    return {"wer": wer(refs, hyps), "cer": cer(refs, hyps), "n": len(refs)}

# evaluate(lambda path: ..., val_pairs)  # wire up per-model transcribe fn once training runs
